**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/watson-2016"], check=True)
    os.chdir("/content/encore/entries/watson-2016")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings checked on CRCNS:fcx-1 (unversioned)

This notebook belongs to the reconciliation of attempt watson-2016-03 with attempt watson-2016-02
(docs/protocol.md, "Repeat attempts"). watson-2016-02 tested the paper on CRCNS:fcx-1 alone and
recorded ten findings; watson-2016-03 tested it on DANDI:000041 version 0.261104.0300 and
CRCNS:fcx-1. watson-2016-03 reports the same problem as earlier findings 2, 4, 7 and 10 (its
findings 7, 6, 8 and 11). The other six, 1, 3, 5, 6, 8 and 9, all about the CRCNS files, are checked
here, and so are the parts of findings 2 and 4 that watson-2016-03 does not report. All of them
concern CRCNS:fcx-1, which is unversioned and is pinned by the MD5 sums of its checksums.md5,
printed below. Each of the 27 session tar.gz files is streamed once from the public bucket with
the entry's helper `watson.py` (its `PrefetchReader`), and only the member list and the
`_WSRestrictedIntervals.mat`, `_BasicMetaData.mat` and `_Motion.mat` files are kept. Nothing is
written to disk. Each section opens with the earlier statement. The conclusions are in the
reconciliation block of `attempts/watson-2016-03.yaml`.

In [ ]:
import io
import re
import tarfile
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import requests

import watson

DOCS = watson.CRCNS_BASE + "docs/"
sums = dict((n, m) for m, n in re.findall(r"(\w{32})\s+(\S+)", requests.get(watson.CRCNS_BASE + "checksums.md5", timeout=60).text))
print(f"CRCNS:{watson.CRCNS_ID} ({watson.CRCNS_VERSION}), doi:{watson.CRCNS_DOI}; MD5 sums of checksums.md5:")
for n in sorted(sums):
    if n.startswith(("data/", "docs/")):
        print(f"  {sums[n]}  {n}")

KEEP = ("_WSRestrictedIntervals.mat", "_BasicMetaData.mat", "_Motion.mat")


def scan(name):
    """Stream one session tar from start to end; keep the member list and three small files."""
    t0 = time.time()
    rdr = watson.PrefetchReader(watson.CRCNS_BASE + f"data/{name}.tar.gz")
    size = rdr.size
    tf = tarfile.open(fileobj=io.BufferedReader(rdr, buffer_size=2**22), mode="r|gz")
    members, files = [], {}
    for m in tf:
        if not m.isfile():
            continue
        base = m.name.split("/")[-1]
        members.append((base, m.size))
        for k in KEEP:
            if base.endswith(k):
                files[k] = tf.extractfile(m).read()
    rdr.close()
    return dict(name=name, gz_bytes=size, members=members, files=files, seconds=time.time() - t0)


t_all = time.time()
with ThreadPoolExecutor(4) as ex:
    S = list(ex.map(scan, watson.CRCNS_SESSIONS))
t_all = time.time() - t_all

In [ ]:
# What was loaded.
print(f"{len(S)} sessions streamed, {sum(s['gz_bytes'] for s in S) / 1e9:.1f} GB compressed, in {t_all:.0f} s with four at a time")
for s in S:
    print(f"  {s['name']}: {s['gz_bytes'] / 1e9:.2f} GB, {len(s['members'])} files, kept {sorted(s['files'])}, {s['seconds']:.0f} s")
assert all(len(s["files"]) == 3 for s in S)

In [ ]:
def intervals(s):
    w = watson._loadmat(s["files"]["_WSRestrictedIntervals.mat"])
    d = {}
    for k in ["REM", "SWSPacket", "MA", "WakeInterruption", "Wake", "Sleep"]:
        v = watson._tp(w[k + "TimePairFormat"])
        d[k] = np.vstack(v) if isinstance(v, list) else v
    ws = w["WakeSleepTimePairFormat"]
    cyc = [np.asarray(c, float) for c in ws] if (isinstance(ws, np.ndarray) and ws.dtype == object) else [np.asarray(ws, float)]
    d["cycles"] = sorted([(c[0], c[1]) for c in cyc], key=lambda c: c[0][0])  # (WAKE [start, stop], SLEEP [start, stop])
    return d


def within(iv, a, b):
    iv = np.asarray(iv, float).reshape(-1, 2)
    return iv[(iv[:, 0] >= a) & (iv[:, 1] <= b)]


for s in S:
    s["I"] = intervals(s)

## Earlier finding 1 (metadata_error)

> The files hold 51 WAKE-SLEEP cycles (WakeSleepTimePairFormat over the 27 sessions), where the
> paper reports 54; the Wake and Sleep intervals are the same 51 pairs, with none unpaired. All
> other counts (995 pE, 126 pI, 27 sessions, 11 rats, 43230 UP and 86839 DOWN states, 5203
> spindles) match the paper exactly.

The WakeSleep pairs of each session, and the Wake and Sleep intervals that are not one of them.

In [ ]:
tot = dict(cycles=0, Wake=0, Sleep=0, unpaired_Wake=0, unpaired_Sleep=0)
for s in S:
    I = s["I"]
    W = {tuple(c[0]) for c in I["cycles"]}
    Sl = {tuple(c[1]) for c in I["cycles"]}
    tot["cycles"] += len(I["cycles"])
    tot["Wake"] += len(I["Wake"])
    tot["Sleep"] += len(I["Sleep"])
    tot["unpaired_Wake"] += sum(tuple(x) not in W for x in I["Wake"])
    tot["unpaired_Sleep"] += sum(tuple(x) not in Sl for x in I["Sleep"])
print(tot, "; paper 54 WAKE-SLEEP episodes")
print("cycles per session:", {s["name"]: len(s["I"]["cycles"]) for s in S})

## Earlier finding 2 (metadata_error), the parts watson-2016-03 does not report

> The paper reports 790 microarousals; 434 MA intervals lie inside SLEEP (2130 MA intervals in the
> files, most outside SLEEP). 345 WakeInterruption intervals inside SLEEP are shorter than the 40 s
> minimum the data description gives for them; none lies between two packets (123 between two REM
> intervals, 80 between REM and a packet), so they are not MAs by the paper's definition either,
> though counted with the MAs they give 779 events. One nonREM packet (14 s) is shorter than the
> 20 s minimum of Table S1.

watson-2016-03 reports the MA count (its finding 7). Checked here: the WakeInterruptions shorter
than 40 s inside the SLEEP of the cycles, with the states on either side of each (within 2 s, the
earlier attempt's rule: P packet, R REM, M MA, W WakeInterruption, - none), the data description's
definition of WakeInterruption, and the packets shorter than 20 s.

In [ ]:
nb, n_short, n_ma = {}, 0, 0
for s in S:
    I = s["I"]
    for _, sl in I["cycles"]:
        n_ma += len(within(I["MA"], *sl))
        seq = sorted([(k, *x) for key, k in (("SWSPacket", "P"), ("REM", "R"), ("MA", "M"), ("WakeInterruption", "W"))
                      for x in within(I[key], *sl)], key=lambda z: z[1])
        for j, (k, a, b) in enumerate(seq):
            if k == "W" and b - a < 40:
                n_short += 1
                prev = seq[j - 1][0] if j > 0 and a - seq[j - 1][2] <= 2 else "-"
                nxt = seq[j + 1][0] if j + 1 < len(seq) and seq[j + 1][1] - b <= 2 else "-"
                nb[prev + "W" + nxt] = nb.get(prev + "W" + nxt, 0) + 1
print(f"MA intervals in the SLEEP of cycles: {n_ma} of {sum(len(s['I']['MA']) for s in S)}")
print(f"WakeInterruptions shorter than 40 s in SLEEP: {n_short}; by neighbours", dict(sorted(nb.items(), key=lambda x: -x[1])))
print(f"  between two packets: {nb.get('PWP', 0)}; between two REM: {nb.get('RWR', 0)}; REM then packet: {nb.get('RWP', 0)}; packet then REM: {nb.get('PWR', 0)}")
print("  counted with the MAs:", n_ma + n_short)
pk = [(s["name"], float(d)) for s in S for d in np.diff(s["I"]["SWSPacket"], axis=1).ravel() if d < 20]
print("packets shorter than 20 s:", pk)
desc = requests.get(DOCS + "crcns_fcx-1_data_description.docx", timeout=60).content
text = re.sub(r"<[^>]+>", "", re.sub(r"</w:p>", "\n", zipfile.ZipFile(io.BytesIO(desc)).read("word/document.xml").decode()))
print("data description:", " ".join(re.search(r"WakeInterruptionTimePairFormat.{0,200}?sleep\.", text, re.S).group(0).split()))

## Earlier finding 3 (metadata_error)

> In Rizzo_022715 and Splinter_020515 a WAKE-SLEEP cycle's WAKE starts before the previous cycle's
> SLEEP ends (WAKE 9919-14394 s against SLEEP 5655-10010 s; WAKE 5335-7210 s against SLEEP
> 3543-5455 s), so the cycles overlap.

Consecutive cycles of each session, by WAKE start, where the next WAKE starts before the SLEEP
before it ends.

In [ ]:
n_ov = 0
for s in S:
    c = s["I"]["cycles"]
    for a, b in zip(c, c[1:]):
        if b[0][0] < a[1][1]:
            n_ov += 1
            print(f"{s['name']}: WAKE {b[0][0]:.0f}-{b[0][1]:.0f} s starts before SLEEP {a[1][0]:.0f}-{a[1][1]:.0f} s ends")
print("overlapping cycle pairs:", n_ov)

## Earlier finding 4 (other, data), the part watson-2016-03 does not report

> Dino_061914 and Dino_062014 each appear as two sessions, ACC and mPFC probes recorded at the same
> time (the data set's WatsonSleepHomestasis2016Table.xlsx lists the same days), each with its own
> state files, and the two scorings of Dino_062014 differ (mean SLEEP per cycle 59.5 min in ACC and
> 27.1 min in mPFC; 41 and 17 packets). State- and LFP-level analyses over the 27 sessions count
> these days twice.

watson-2016-03 reports the two folders of each day (its finding 6). Checked here: the cycles and
packets of the two scorings of each day.

In [ ]:
for n in ["Dino_061914_ACC", "Dino_061914_mPFC", "Dino_062014_ACC", "Dino_062014_mPFC"]:
    I = next(s for s in S if s["name"] == n)["I"]
    sl = [c[1] for c in I["cycles"]]
    print(f"{n}: {len(sl)} cycles, SLEEP {[f'{a:.0f}-{b:.0f}' for a, b in sl]} s, mean SLEEP {np.mean([b - a for a, b in sl]) / 60:.1f} min, "
          f"{len(I['SWSPacket'])} packets ({sum(len(within(I['SWSPacket'], *x)) for x in sl)} in SLEEP)")

## Earlier finding 5 (metadata_error)

> The data description says the motion signal in _Motion.mat is z-scored; it is non-negative in
> every session, with per-session means from 0.55 to 4076, and BWRat19_032513 holds NaN values.

The `motion` field of each session's `_Motion.mat`, and the data description's text for it.

In [ ]:
mo = {}
for s in S:
    m = np.atleast_1d(np.asarray(watson._loadmat(s["files"]["_Motion.mat"])["motiondata"].motion, float))
    mo[s["name"]] = m
print("data description:", re.search(r"\.motion[^\n]*", text).group(0))
print("sessions with a negative value:", [k for k, m in mo.items() if np.nanmin(m) < 0], "; smallest value:", min(np.nanmin(m) for m in mo.values()))
nan = {k: f"{int(np.isnan(m).sum())} of {len(m)}" for k, m in mo.items() if np.isnan(m).any()}
print("sessions with NaN:", nan)
means = {k: float(np.mean(m)) for k, m in mo.items() if not np.isnan(m).any()}
print(f"per-session mean, sessions without NaN: {min(means.values()):.3g} to {max(means.values()):.4g}; "
      f"per-session SD: {min(np.nanstd(m) for m in mo.values()):.3g} to {max(np.nanstd(m) for m in mo.values()):.4g}")
print("NaN sessions, mean over the other samples:", {k: round(float(np.nanmean(mo[k])), 3) for k in nan})

## Earlier finding 6 (metadata_error)

> RecordingFileIntervals in _BasicMetaData.mat ends at infinity in the three sessions of rat
> JennBuzsaki22, and the UP/DOWN file is named _UPDOWNINtervals.mat, where the description says
> _UPDOWNInts.mat and _UPDOWNIntervals.mat.

`RecordingFileIntervals` of each session, the names of the UP/DOWN files in the tars, and the names
the data description gives.

In [ ]:
for s in S:
    r = np.atleast_2d(np.asarray(watson._loadmat(s["files"]["_BasicMetaData.mat"])["bmd"].RecordingFileIntervals, float))
    if not np.all(np.isfinite(r)):
        print(f"{s['name']}: RecordingFileIntervals {r.tolist()}")
print("sessions with every bound finite:", sum(np.all(np.isfinite(np.asarray(watson._loadmat(s['files']['_BasicMetaData.mat'])['bmd'].RecordingFileIntervals, float))) for s in S))
names = sorted({m.split(s["name"])[-1] for s in S for m, _ in s["members"] if "updown" in m.lower()})
print("UP/DOWN file names in the tars (after the session name):", names, "; sessions with one:",
      sum(any("updown" in m.lower() for m, _ in s["members"]) for s in S))
print("names in the data description:", sorted(set(re.findall(r"basename(_UPDOWN\w*\.mat)", text))))

## Earlier finding 8 (metadata_insufficient)

> Scored WAKE exists only as the WAKE of a WAKE-SLEEP cycle, so the WAKE that follows a SLEEP is in
> most cycles absent or hours later (23 of 51 cycles have a later scored WAKE, 6 within 1 s). The
> 5 min post-sleep WAKE windows of the paper cannot be located for most cycles.

Whether every Wake interval is a cycle's WAKE (finding 1 above), and for each cycle the first
scored WAKE of its session that starts at or after the end of its SLEEP (0.5 s slack for whole
seconds).

In [ ]:
gaps, n_cyc = [], 0
for s in S:
    I = s["I"]
    wakes = np.array([c[0] for c in I["cycles"]])
    for _, sl in I["cycles"]:
        n_cyc += 1
        nxt = wakes[wakes[:, 0] >= sl[1] - 0.5]
        if len(nxt):
            gaps.append(nxt[:, 0].min() - sl[1])
gaps = np.array(gaps)
print(f"cycles with a later scored WAKE: {len(gaps)} of {n_cyc}; within 1 s: {int(np.sum(gaps <= 1))}; "
      f"the others start {gaps[gaps > 1].min():.0f} to {gaps.max():.0f} s after SLEEP ends")

## Earlier finding 9 (other, data)

> Each session is one tar.gz of 0.9 to 6 GB, and the order of members differs between files (the
> LFP .eeg comes before the spike and state files in some), so reading any one small file needs the
> whole archive streamed. Reading all 27 takes 7 to 9 minutes from the bucket.

The tar sizes, the position of the .eeg among the members, and the time of the read above.

In [ ]:
small = ("_WSRestrictedIntervals.mat", "_BasicMetaData.mat", "_SStable.mat", "_CellIDs.mat")
after = []
for s in S:
    order = [m for m, _ in s["members"]]
    e = next(i for i, m in enumerate(order) if m.endswith(".eeg"))
    late = [m.split(s["name"])[-1] for m in order[e + 1:] if m.endswith(small)]
    if late:
        after.append((s["name"], late))
print(f"tar.gz sizes {min(s['gz_bytes'] for s in S) / 1e9:.2f} to {max(s['gz_bytes'] for s in S) / 1e9:.2f} GB, "
      f"{sum(s['gz_bytes'] for s in S) / 1e9:.1f} GB in all; files per tar {min(len(s['members']) for s in S)} to {max(len(s['members']) for s in S)}")
print(f"sessions whose .eeg comes before a spike, cell or state file: {len(after)} of {len(S)}")
for n, late in after:
    print(f"  {n}: after the .eeg {late}")
print(f"sessions with the .eeg last: {sum(s['members'][-1][0].endswith('.eeg') for s in S)}")
print(f"the read above took {t_all:.0f} s ({t_all / 60:.1f} min) with four sessions at a time")